# SlaTriage — phần còn lại (B, C, D, E_noR, G)

Không train. Không chạy lại E nếu đã đủ 343. Ghi từng contract lên Drive, đứt tab chạy lại cell này là tiếp.

1. Runtime → GPU T4 (hoặc L4/A100 càng tốt).
2. `slatriage_adapters.zip` nằm trên Drive (đủ R/A/C/J).
3. Giữ tab mở. Sau B: `pred_B.jsonl` phải `n=343 sources=112`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile
from pathlib import Path
assert __import__('torch').cuda.is_available(), 'Runtime → Change runtime type → T4 GPU, rồi Restart'

REPO = 'https://github.com/kham3465/Qu-n-Tr-An-Ninh-M-ng.git'
ROOT = Path('/content/slatriage')
if not (ROOT / '.git').exists():
    !git clone --depth 1 {REPO} {ROOT}
!git -C {ROOT} fetch origin master
!git -C {ROOT} reset --hard origin/master

ad = Path('/content/adapters')
if not (ad / 'R' / 'adapter_config.json').exists():
    zips = list(Path('/content/drive/MyDrive').rglob('slatriage_adapters.zip'))
    assert zips, 'Upload slatriage_adapters.zip (R/A/C/J) lên Drive'
    ad.mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(zips[0]).extractall(ad)
    if not (ad / 'R' / 'adapter_config.json').exists():
        nested = list(ad.rglob('R/adapter_config.json'))
        assert nested, zips[0]
        ad = nested[0].parent.parent

out = Path('/content/drive/MyDrive/slatriage_reports')
out.mkdir(parents=True, exist_ok=True)
os.environ['SLATRIAGE_ADAPTERS'] = str(ad)
os.environ['SLATRIAGE_PRED_DIR'] = str(out)
os.environ['SLATRIAGE_CONFIGS'] = 'remaining'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'
print('adapters', ad)
print('pred_dir', out)
os.chdir(ROOT)
!pip install -q -r requirements-train.txt
!python scripts/colab_eval_rest.py